# Saboteur finder demo -- the formal feature blacklist

The similarity-metric study kept finding individual columns inside an otherwise-useful
feature block that **actively hurt** a yardstick. Removing them **raises** the score.
Those columns are *saboteurs*; the set is the *blacklist* that `find_best_hybrid`
now accepts via its `feature_blacklist` argument.

This notebook runs the standalone finder `blacklist.identify_saboteurs` on the
**herg_ki** `desc` block (208 RDKit physicochemical descriptors), judged by the
**structural fidelity** yardstick -- Spearman(pairwise Euclidean distance, MCS distance)
over the cached MCS pairs.

The finder is cross-validated: every drop decision is made on TRAIN scaffold folds and
re-measured on HELD-OUT folds, and a column only reaches the blacklist if it is picked
in a majority of folds (stability guard) with a positive mean held-out gain (threshold
guard). Expected saboteurs: `fr_Ar_OH`, `fr_HOCCN`, `fr_nitrile`.

In [ ]:
import os, sys
import numpy as np

# this notebook lives in tests/experiments/similarity_metrics/
HERE = os.path.dirname(os.path.abspath('saboteur_finder_demo.ipynb'))
if HERE not in sys.path:
    sys.path.insert(0, HERE)

from rdkit.Chem import Descriptors
from _data import load_prepared
from blacklist import identify_saboteurs

# desc column names, in the exact order build_moleculeace.py wrote the desc matrix
DESC_NAMES = [n for n, _ in Descriptors._descList]
print('desc feature names:', len(DESC_NAMES))

In [ ]:
# load herg_ki (used whole, n=1415) and the cached MCS pairs
p = load_prepared('herg_ki')
desc = p['desc']
y = p['y']
groups = p['groups']
mcs_pairs = np.load(os.path.join(HERE, 'mcs_pairs_herg_ki.npz'))

print(f"n molecules      : {p['n']}")
print(f"scaffold groups  : {p['n_scaffolds']}")
print(f"desc block shape : {desc.shape}")
print(f"MCS pairs        : {len(mcs_pairs['I'])}")
assert desc.shape[1] == len(DESC_NAMES)

In [ ]:
# run the saboteur finder: greedy-backward ablation on held-out scaffold folds,
# structural-fidelity yardstick.
result = identify_saboteurs(
    rep_matrix=desc,
    feature_names=DESC_NAMES,
    y=y,
    groups=groups,
    structural_pairs=mcs_pairs,
    yardstick='structural',
    n_splits=5,
    verbose=True,
)
print('\nyardstick        :', result['yardstick'])
print('stability guard  : dropped in >=', result['min_folds'], 'of', result['n_splits'], 'folds')
print('threshold guard  : mean held-out gain >', result['min_heldout_gain'])

In [ ]:
# the ranked blacklist (features that cleared BOTH guards)
print('FORMAL BLACKLIST (saboteurs):')
print(f"  {'feature':22s} {'folds':>5} {'mean_heldout_gain':>18} {'mean_train_gain':>16}")
for r in result['blacklist']:
    print(f"  {r['name']:22s} {r['n_folds']:>5} {r['mean_heldout_gain']:>18.5f} {r['mean_train_gain']:>16.5f}")
print('\nblacklist_cols (feed to find_best_hybrid feature_blacklist):', result['blacklist_cols'])
print('blacklist_names:', result['blacklist_names'])

In [ ]:
# full ranking (every feature ever dropped in any fold), top 15
print('TOP 15 candidates by (stability, mean held-out gain):')
print(f"  {'feature':22s} {'folds':>5} {'mean_heldout_gain':>18} {'blacklisted':>11}")
for r in result['ranked'][:15]:
    print(f"  {r['name']:22s} {r['n_folds']:>5} {r['mean_heldout_gain']:>18.5f} {str(r['in_blacklist']):>11}")

In [ ]:
# before / after: held-out structural fidelity with the FULL desc block vs the
# blacklist removed, averaged over the same scaffold folds.
print(f"structural fidelity BEFORE (full desc)     : {result['baseline_score']:+.4f}")
print(f"structural fidelity AFTER  (blacklist gone) : {result['ablated_score']:+.4f}")
print(f"delta (higher = better)                     : {result['delta']:+.4f}")

expected = {'fr_Ar_OH', 'fr_HOCCN', 'fr_nitrile'}
found = set(result['blacklist_names'])
print('\nexpected saboteurs found:', sorted(expected & found))
print('other saboteurs found   :', sorted(found - expected))